# NN03: 16 units with fixed rate 0.3

Executable notebook copy of `Models/NN03_model.py`. The class definition below is copied from that file and uses the shared NN01 training implementation. Ten seeds, SGD, cross-entropy, whole-election early stopping, patience 20. Set `TRAIN_MODEL=True` to fit the 2019 development example; by default the notebook displays the saved seven-election results.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Models/NN01_model.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PACKAGE = ROOT / 'Analysis and model development/05_nn_first_development'
RESULTS = PACKAGE / 'results'


## Model definition

In [2]:
"""NN03: 16 units fixed rate 0.3. Ten seeds; latest whole election held out; patience 20.

Uses the shared implementation in NN01_model without changing its configuration.
See Analysis and model development/05_nn_first_development/.
"""
from Models.NN01_model import NeuralNetworkModel as BaseNeuralNetworkModel

HIDDEN_SIZES = (16,)
LEARNING_RATES = (0.3,)
CLASS_LABELS = ('con', 'lab', 'lib', 'natSW', 'oth')


class NeuralNetworkModel(BaseNeuralNetworkModel):
    def __init__(self):
        super().__init__(hidden_sizes=HIDDEN_SIZES, learning_rates=LEARNING_RATES,
                         name='NN03: 16 units fixed rate 0.3', class_labels=CLASS_LABELS)


In [3]:
model = NeuralNetworkModel()
print(model.name)
print('Hidden layers:', model.hidden_sizes)
print('Learning rates:', model.learning_rates)
print('Classes:', model.class_labels)

NN03: 16 units fixed rate 0.3
Hidden layers: (16,)
Learning rates: (0.3,)
Classes: ('con', 'lab', 'lib', 'natSW', 'oth')


## Saved historical development results

In [4]:
historical=pd.read_csv(RESULTS/'election_diagnostics.csv')
historical=historical.loc[(historical.architecture=='16') & (historical.policy=='fixed_0.3')]
display(historical[['evaluation_year','learning_rate','accuracy','changed_accuracy','overall_changed_score','log_loss','correct_changes','false_changes_on_held_seats']])

,evaluation_year,learning_rate,accuracy,changed_accuracy,overall_changed_score,log_loss,correct_changes,false_changes_on_held_seats
1,1997,0.3,0.684867,0.050000,0.367434,1.338150,8,4
9,2001,0.3,0.945398,0.166667,0.556032,0.205561,4,15
17,2005,0.3,0.802548,0.631579,0.717063,0.443566,36,103
25,2010,0.3,0.837025,0.875000,0.856013,0.403291,98,89
33,2015,0.3,0.775316,0.146789,0.461053,0.828193,16,49
41,2017,0.3,0.898734,0.149254,0.523994,0.283829,10,7
49,2019,0.3,0.911392,0.434211,0.672801,0.222753,33,13


## Optional fit: train through 2015, validate on 2017, evaluate on 2019
Training uses the retained historical snapshot only. The whole 2017 election selects checkpoints; evaluation outcomes are not used to choose the rate.

In [5]:
TRAIN_MODEL = False
if TRAIN_MODEL:
    import numpy as np
    import torch
    from sklearn.metrics import accuracy_score, log_loss
    torch.set_num_threads(1)
    data=pd.read_csv(PACKAGE/'data/train.csv')
    training=data.loc[data.election<=2017]
    evaluation=data.loc[data.election==2019]
    model.train(training)
    probabilities=model.predict_proba(evaluation)
    predictions=model.predict(evaluation)
    changed=evaluation.previous_winner.notna() & evaluation.winner.ne(evaluation.previous_winner)
    print('Selected rate:',model.selected_learning_rate)
    print('Accuracy:',accuracy_score(evaluation.winner,predictions))
    print('Changed-seat accuracy:',accuracy_score(evaluation.loc[changed,'winner'],predictions[changed]))
    print('Log loss:',log_loss(evaluation.winner,probabilities,labels=model.classes_))
    display(model.learning_rate_summary)
else:
    print('Saved results displayed. Set TRAIN_MODEL=True to run the full ten-seed procedure.')

Saved results displayed. Set TRAIN_MODEL=True to run the full ten-seed procedure.
